# M0-04 · MC-OCR data analysis

Notebook này dùng để kiểm tra nhanh dataset Vietnamese Receipts MC-OCR 2021, xem phân bố nhãn/chất lượng ảnh và tạo dev-set 20-30 mẫu sạch cho Node 1 Extractor.

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 160)

repo_root = Path.cwd()
if not (repo_root / 'data').exists() and any(parent.name == 'do-an-ptit-d22' for parent in Path.cwd().parents):
    repo_root = next(parent for parent in Path.cwd().parents if parent.name == 'do-an-ptit-d22')

data_root = repo_root / 'data' / 'raw' / 'mc_ocr_2021'
print('repo_root =', repo_root)
print('data_root =', data_root)
print('data exists =', data_root.exists())

## 1. Tìm CSV và thư mục ảnh train

In [ ]:
csv_paths = sorted(data_root.rglob('mcocr_train_df.csv'))
train_image_dirs = sorted(path for path in data_root.rglob('*') if path.is_dir() and path.name.lower() == 'train_images')

print('CSV candidates:')
for path in csv_paths:
    print('-', path.relative_to(repo_root))

print('\ntrain_images candidates:')
for path in train_image_dirs:
    print('-', path.relative_to(repo_root))

train_csv = csv_paths[0]
df = pd.read_csv(train_csv)
df.shape, df.columns.tolist()

## 2. Xem vài dòng dữ liệu gốc

In [ ]:
df.head(3)

## 3. Chất lượng ảnh và số dòng text

In [ ]:
def repair_mojibake(text):
    markers = ('Ã', 'Â', 'Ä', 'Æ', 'Ð', 'áº', 'á»')
    has_c1_control = any('\u0080' <= char <= '\u009f' for char in text)
    if not has_c1_control and not any(marker in text for marker in markers):
        return text
    try:
        return text.encode('latin1').decode('utf-8')
    except UnicodeError:
        return text

def split_mcocr_field(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return []
    return [repair_mojibake(part.strip()) for part in text.split('|||') if part.strip()]

work = df.copy()
work['labels_list'] = work['anno_labels'].apply(split_mcocr_field)
work['texts_list'] = work['anno_texts'].apply(split_mcocr_field)
work['label_count'] = work['labels_list'].apply(len)
work['text_line_count'] = work['texts_list'].apply(len)

work[['anno_image_quality', 'anno_num', 'label_count', 'text_line_count']].describe()

In [ ]:
ax = work['anno_image_quality'].hist(bins=30, figsize=(8, 4))
ax.set_title('MC-OCR train image quality distribution')
ax.set_xlabel('anno_image_quality')
ax.set_ylabel('receipt count')
plt.show()

## 4. Phân bố nhãn nghiệp vụ

In [ ]:
label_counter = Counter(label.strip().upper() for labels in work['labels_list'] for label in labels)
label_counts = pd.DataFrame(label_counter.most_common(), columns=['label', 'count'])
label_counts

In [ ]:
top_labels = label_counts.head(20).sort_values('count')
ax = top_labels.plot.barh(x='label', y='count', figsize=(8, 5), legend=False)
ax.set_title('Top MC-OCR labels')
ax.set_xlabel('count')
plt.show()

## 5. Lọc ứng viên dev-set sạch

Tiêu chí mặc định:
- `anno_image_quality >= 0.85`
- có `SELLER`, `ADDRESS`, `TIMESTAMPS`
- ưu tiên có `TOTAL_COST`
- ảnh tồn tại trong `train_images/`

In [ ]:
image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_index = {}
for image_root in train_image_dirs:
    for image_path in image_root.rglob('*'):
        if image_path.is_file() and image_path.suffix.lower() in image_suffixes:
            image_index.setdefault(image_path.name, image_path)

required = {'SELLER', 'ADDRESS', 'TIMESTAMPS'}
preferred = {'TOTAL_COST'}

def has_required(labels):
    label_set = {label.strip().upper() for label in labels}
    return required.issubset(label_set)

work['image_exists'] = work['img_id'].map(lambda img_id: img_id in image_index)
work['has_required'] = work['labels_list'].apply(has_required)
work['has_total_cost'] = work['labels_list'].apply(lambda labels: bool(preferred & {label.strip().upper() for label in labels}))

candidates = work[(work['anno_image_quality'] >= 0.85) & work['has_required'] & work['image_exists']].copy()
candidates['score'] = candidates['anno_image_quality'] + candidates['has_total_cost'].astype(int) * 0.05 + candidates['text_line_count'].clip(upper=80) * 0.0005
selected_preview = candidates.sort_values(['score', 'anno_image_quality', 'text_line_count'], ascending=False).head(30)

selected_preview[['img_id', 'anno_image_quality', 'anno_num', 'text_line_count', 'has_total_cost', 'anno_labels']].head(30)

## 6. Tạo dev-set bằng script chuẩn

Cell dưới sẽ copy ảnh đã chọn vào `data/processed/m0_04_mcocr_devset/images/`, tạo `samples.jsonl` cho pipeline local, và ghi manifest tracked ở `experiments/results/m0_04_mcocr_devset_manifest.csv`.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'prepare_mcocr_devset.py'),
    '--sample-size', '30',
    '--quality-threshold', '0.85',
]
subprocess.run(cmd, cwd=repo_root, check=True)

## 7a. Build Node 1 Extractor inputs and chat requests

Cell dưới tạo input text và request JSONL cho Node 1 Extractor. Request chỉ chứa metadata + OCR text, không đưa nhãn MC-OCR vào prompt để tránh leak đáp án.

In [ ]:
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'build_node1_requests.py'),
    '--model', 'qwen2.5:7b-instruct',
]
subprocess.run(cmd, cwd=repo_root, check=True)

## 7b. Smoke-test Node 1 với Ollama

Cell dưới gọi một mẫu qua endpoint OpenAI-compatible của Ollama, kiểm tra `invoice_extraction.v1` và nạp kết quả hợp lệ vào SQLite. Đây chỉ là kiểm thử chức năng local, không dùng làm số liệu hiệu năng của đồ án. Cần tải model trước bằng `ollama pull qwen2.5:7b-instruct`.

In [ ]:
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_node1_extractor.py'),
    '--limit', '1',
    '--run-id', 'm0_04_node1_ollama_notebook_smoke',
    '--output-dir', str(repo_root / 'data' / 'processed' / 'm0_04_mcocr_devset' / 'node1' / 'notebook_smoke'),
    '--report', str(repo_root / 'data' / 'processed' / 'm0_04_mcocr_devset' / 'node1' / 'notebook_smoke_summary.json'),
    '--replace-run',
]
subprocess.run(cmd, cwd=repo_root, check=True)